# Distance log data processing 
This file goes through the process of reading the pre-processed textract APIs csv distance output file, extracting and transforming the data and saving the data into the target S3 folder in the parquet format partitioned by year and month. The pandas library has been used for faster delivery as it is a small file. In production this can be replicated using the spark dataframe for faster processing of larger files

Importing the libraries

In [16]:
import pandas as pd
import numpy as np
import boto3


Initialising the schema that is expected at the warehouse layer and accessing the files from S3 input path

In [17]:
dfs = []
schema = ["trip_date","start_odometer","end_odometer","distance","AB_kms", "BC_kms", "SK_kms", "MB_kms", "ON_kms",
       "QC_kms", "YT_kms","total_fuel","trip_origin_city","trip_origin_province","trip_destination_city","trip_destination_province"]

Function to perform the transformation for data cleaning

In [18]:
def transformation_process(file_path):
    
    df = pd.read_csv(file_path)
    df.columns = columns_org
    
    # Removing undesired symbols and cleaning the data to extract only values and exclude the confidence scores
    df = df.replace({"^'":""}, regex=True)
    row_count = df[df["'DATE"]=="Confidence Scores % (Table Cell)"].index[0]
    data = df.head(row_count)
    
    
    # Adding total_fuel column
    data['total_fuel'] = data.loc[:,["'AB Fuel", "'BC Fuel", "'SK Fuel", "'MB Fuel", "'ON Fuel", "'QC Fuel"]].sum(axis=1)
    data = data.replace({"":np.nan}, regex=True)
    keep_columns = ["'DATE", "'START KM", "'STARTING POINT", "'DESTINATION", "'END KM","'TOTAL KM","'AB KMs", "'BC KMs", "'SK KMs", "'MB KMs", "'ON KMs",
       "'QC KMs", "'YT KMs", "total_fuel"]
    
    data = data.dropna(subset=["'DATE", "'START KM", "'STARTING POINT"])

    data = data[keep_columns]
    
    new_cols_name = ["trip_date","start_odometer","trip_origin","trip_destination","end_odometer","distance","AB_kms", "BC_kms", "SK_kms", "MB_kms", "ON_kms",
       "QC_kms", "YT_kms","total_fuel"]
    
    data.columns=new_cols_name
    
    # Refractoring city and province level columns
    data[['trip_origin_city', 'trip_origin_province']] = data["trip_origin"].str.split(', ', expand=True)
    data[['trip_destination_city', 'trip_destination_province']] = data["trip_destination"].str.split(', ', expand=True)
    
    data['trip_origin_city'] = data["trip_origin_city"].str.split(' to').str[0]
    data['total_fuel'] = data['total_fuel'].fillna(0)
    
    data=data.drop(columns = ['trip_origin','trip_destination'])
    
    # Filling null values 
    data[["AB_kms", "BC_kms", "SK_kms", "MB_kms", "ON_kms",
       "QC_kms", "YT_kms"]] = data[["AB_kms", "BC_kms", "SK_kms", "MB_kms", "ON_kms",
       "QC_kms", "YT_kms"]].fillna(0)
    
    data = data[schema]
    dfs.append(data)

Reading files from S3 and processing

In [19]:
columns_org = ["'DATE", "'START KM", "'STARTING POINT", "'DESTINATION", "'END KM",
       "'TOTAL KM", "'AB KMs", "'BC KMs", "'SK KMs", "'MB KMs", "'ON KMs",
       "'QC KMs", "'YT KMs", "'AB Fuel", "'BC Fuel", "'SK Fuel", "'MB Fuel",
       "'ON Fuel", "'QC Fuel", 'Unnamed: 19']

s3 = boto3.resource('s3')
bucket = s3.Bucket('cn01-project-pre-processed-files-205096516800-us-east-2-an')
for obj in bucket.objects.filter(Prefix= "input_files/distancelog_files"):
    key = obj.key
    if key == "input_files/distancelog_files/":
        continue
    
    
    file_path = "s3://" + bucket.name + "/" + str(key)
    transformation_process(file_path)
    
    


<stdin>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<stdin>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<stdin>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<stdin>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFra

Performing data formatting and ensuring undesired symbols are removed from values

In [ ]:
final = pd.concat(dfs)

final['vin_number'] = pd.NA
final['trip_date'] = pd.to_datetime(final['trip_date'])
final['year'] = final['trip_date'].dt.year
final['month'] = final['trip_date'].dt.month
final['trip_date'] = pd.to_datetime(final['trip_date']).dt.date

final['start_odometer'] = final['start_odometer'].astype(float)
final['end_odometer'] = final['end_odometer'].astype(float)
final['distance'] = final['distance'].astype(float)
final['AB_kms'] = final['AB_kms'].astype(float)
final['BC_kms'] = final['BC_kms'].astype(float)
final['SK_kms'] = final['SK_kms'].astype(float)
final['MB_kms'] = final['MB_kms'].astype(float)
final['ON_kms'] = final['ON_kms'].astype(float)
final['QC_kms'] = final['QC_kms'].astype(float)
final['YT_kms'] = final['YT_kms'].astype(float)
final['total_fuel'] = final['total_fuel'].astype(float)

final['trip_destination_city'] = final['trip_destination_city'].str.lower().str.strip()
final['trip_origin_city'] = final['trip_origin_city'].str.lower().str.strip()


final['trip_destination_province'] = final['trip_destination_province'].fillna("AB")
final['trip_origin_province'] = final['trip_origin_province'].fillna("AB")


final.shape

# Null values in vin_number
final.isnull().sum()

# 17 duplicates dropped
final.drop_duplicates(inplace=True)


print("Data process complete")

Data process complete


In [21]:
# Showing all columns
pd.set_option('display.max_columns', None)
print(final.head(10))

    trip_date  start_odometer  end_odometer  distance  AB_kms  BC_kms  SK_kms  \
0  2022-02-05        771515.0      772250.0     735.0     0.0   735.0     0.0   
1  2022-03-05        772250.0      772750.0     500.0   450.0    50.0     0.0   
2  2022-04-05        772750.0      773800.0    1050.0  1050.0     0.0     0.0   
3  2022-05-05        773800.0      774430.0     630.0   550.0    80.0     0.0   
4  2022-06-05        774430.0      775250.0     820.0   600.0   220.0     0.0   
5  2022-09-05        775250.0      776280.0    1030.0  1030.0     0.0     0.0   
6  2022-10-05        776280.0      777300.0    1020.0  1020.0     0.0     0.0   
7  2022-11-05        777300.0      778050.0     750.0   600.0   150.0     0.0   
8  2022-12-05        778050.0      778950.0     900.0   700.0   200.0     0.0   
9  2022-05-15        778950.0      779750.0     800.0   450.0   350.0     0.0   

   MB_kms  ON_kms  QC_kms  YT_kms  total_fuel trip_origin_city  \
0     0.0     0.0     0.0     0.0       55

In [4]:
# final.to_parquet(path="s3://cn01-project-output-205096516800-us-east-2-an/output_parquet_files/distance_logs/",partition_cols=['year','month'])
